# 09 · Citation practice: what inspectors write down, and how they flag it

Notebook 08 shows inspectors differ by about 11 points in the scores they give the same
property. This notebook looks inside the inspection at where that comes from.

- **What gets cited.** Citations per inspection by tier (the pipeline's rubric-aligned tier
  coding: life safety, critical violation, health hazard, habitability, pool, administrative,
  exterior nuisance), by inspector, holding the property fixed.
- **Finding versus grading.** How much of an inspector's score effect is explained by how many
  citations they write, and how much remains after that (how hard they grade what they cite).
- **Flagging.** Among citations of the city's life-safety items, how often each inspector
  marks a life hazard.
- **Noted but not flagged.** Inspector notes that describe a life-hazard condition (no heat, no
  AC, no hot water, sewage, exposed wiring) on citations not flagged as life hazards.

Each observation row with a checklist item is one citation. Rows with no item carry no
notes, so every note belongs to a citation. The raw export has no per-item severity or
points. Notes can contain unit numbers, so only counts and rates are printed.

## Findings

Run 2026-10-05. Outputs are cleared before commit; the numbers below are from that run.

**Inspectors differ in how much they write down, and in which checklist item they use for
the same condition, which decides whether a unit is scored zero.**

- **Citation volume varies widely.** There are 38,155 citations on 4,272 inspections, and
  30.7% of scored inspections have no citation at all. Across 36 inspectors with 30+
  inspections:
  - the share of inspections with no citation runs 6% to 62% (10th to 90th percentile);
  - citations per inspection run 1.6 to 11.9;
  - holding the property fixed, inspector effects on citations per inspection have an SD of
    4.5.
- **Some inspectors rarely cite whole categories.** 8 of the 36 cite a life-safety item in
  fewer than 2% of their inspections. 4 almost never cite pool safety and 3 almost never cite
  administrative items.
- **Finding versus grading.** An inspector's score effect correlates −0.67 with their
  citation-count effect. Removing what citation counts explain lowers the SD of inspector
  score effects from 4.85 to 3.59 points, so how much an inspector cites accounts for about
  45% of the spread. The rest is how hard they grade what they cite. The export has no
  per-item points to split that further.
- **Life-hazard flagging depends on the checklist item.** Only 32% of the 719 life-safety
  citations are flagged as life hazards:
  - **specific items** ("no hot water", "air conditioning not in operating condition",
    "exposed electrical") are flagged nearly 100% of the time;
  - **general items** (the water-heater pressure valve, "provide and maintain… air
    conditioning / heating") are flagged 1–7% of the time.

  Among 19 inspectors with 10+ life-safety citations, the share flagged runs 0% to 51%, and
  3 never flag one.
- **Same condition, different item.**

  | Condition | Citations | Share using the life-hazard item |
  |---|---|---|
  | Air conditioning | 132 | 45% |
  | Heating | 36 | 25% |
  | Hot water | 149 | 58% |

  - **Not a checklist change:** both items were in use every year from FY2021-22.
  - **Drift over time:** for AC, use of the life-hazard item fell from 67% (FY2021-22) to 20%
    (FY2024-25).
  - **By inspector:** among 21 inspectors with 5+ such citations, use of the life-hazard
    item runs 0% to 78%, and 4 always use the general item.

  A life-hazard item zeroes the unit, so at a small property this one choice can move the
  score 13 points (notebook 10).
- **Hazards noted but not flagged.** 105 inspections carry an inspector note describing no
  heat, no AC, no hot water, sewage or exposed wiring. 57% of them have no life hazard
  flagged anywhere in the inspection. Their median score is 85, and 32% of the unflagged ones
  score 90 or above. Counts are small and keyword matching is approximate.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

import holdout as h
import reliability as rl
import temporal as t

pd.set_option("display.float_format", "{:.3f}".format)
insp = t.load_inspections()
insp = insp[insp.inspector >= 0]
obs = rl.tier_observations()
cited = obs[obs.subject.fillna("").str.strip() != ""].copy()

TIERS = ["life_safety", "critical_violation", "health_hazard", "habitability",
         "pool_safety", "pool_admin", "administrative", "exterior_nuisance"]
counts = cited.pivot_table(index="sr_number", columns="tier", values="subject", aggfunc="size", fill_value=0)
counts = counts.reindex(columns=TIERS, fill_value=0)
counts["total"] = counts.sum(axis=1)
d = insp.join(counts, on="sr_number").fillna({c: 0 for c in TIERS + ["total"]})
n_by = d.inspector_id.value_counts()
big = n_by[n_by >= 30].index
print(f"{len(cited):,} citations on {cited.sr_number.nunique():,} inspections; "
      f"{(d.total == 0).mean():.1%} of scored inspections have no citation")

## 1. What gets cited, by inspector

In [ ]:
raw = d[d.inspector_id.isin(big)].groupby("inspector_id")[TIERS + ["total"]].mean()
raw["no citation"] = d[d.inspector_id.isin(big)].groupby("inspector_id").total.apply(lambda s: (s == 0).mean())
print(f"citations per inspection, {len(big)} inspectors with 30+ inspections (raw, not property-adjusted):")
print(raw.describe(percentiles=[.1, .5, .9]).loc[["10%", "50%", "90%", "min", "max"]].round(2).to_string())

share_citing = d[d.inspector_id.isin(big)].groupby("inspector_id")[TIERS].agg(lambda s: (s > 0).mean())
print("\ninspectors (of", len(big), ") who cite each tier in fewer than 2% of their inspections:")
print((share_citing < 0.02).sum().to_string())

In [ ]:
# Property-adjusted: citations = property + inspector + year
adj = {}
for col in ["total"] + TIERS:
    e = rl.two_way_effects(d, y=col)["rater"].reindex(d.inspector.drop_duplicates())
    ids = d.drop_duplicates("inspector").set_index("inspector").inspector_id
    e.index = e.index.map(ids)
    adj[col] = e.reindex(big)
adj = pd.DataFrame(adj)
print("property-adjusted inspector effects on citations per inspection (relative to the average inspector):")
print(adj.describe(percentiles=[.1, .9]).loc[["std", "10%", "90%"]].round(2).to_string())

## 2. Finding versus grading

In [ ]:
X = np.column_stack([np.ones(len(d))] + [d[c].to_numpy(float) for c in TIERS])
beta, *_ = np.linalg.lstsq(X, d.score.to_numpy(float), rcond=None)
print("score points per citation, by tier (pooled OLS):", dict(zip(TIERS, beta[1:].round(2))))
d["score_given_citations"] = d.score - X[:, 1:] @ beta[1:]

e_score = rl.two_way_effects(d, y="score")["rater"]
e_resid = rl.two_way_effects(d, y="score_given_citations")["rater"]
e_total = rl.two_way_effects(d, y="total")["rater"]
keep = d[d.inspector_id.isin(big)].inspector.unique()
print(f"SD of inspector score effects: raw {e_score.reindex(keep).std():.2f} points; "
      f"after removing what their citation counts explain {e_resid.reindex(keep).std():.2f} points")
print(f"correlation of an inspector's score effect with their citation-count effect: "
      f"{np.corrcoef(e_score.reindex(keep), e_total.reindex(keep))[0, 1]:.2f}")

## 3. Flagging life hazards on life-safety items

In [ ]:
ls = cited[cited.tier == "life_safety"].merge(insp[["sr_number", "inspector_id"]], on="sr_number")
print(f"life-safety citations: {len(ls):,}; flagged as life hazard: {ls.life_hazard.mean():.1%}")
by = ls.groupby("inspector_id").life_hazard.agg(["size", "mean"])
by = by[by["size"] >= 10].sort_values("mean")
print(f"inspectors with 10+ life-safety citations: {len(by)}")
print(f"  share flagged, percentiles 10/50/90: {by['mean'].quantile([.1, .5, .9]).round(2).tolist()}")
print(f"  inspectors who flag none: {(by['mean'] == 0).sum()}; who flag all: {(by['mean'] == 1).sum()}")
item = ls.assign(item=ls.subject.str.slice(0, 70)).groupby("item").life_hazard.agg(["size", "mean"])
print("\nmost common life-safety items, share flagged:")
print(item[item["size"] >= 20].sort_values("size", ascending=False).round(2).to_string())

### Same condition, two checklist items

For AC, heat and hot water the checklist carries both a specific item that is always
flagged a life hazard (unit scored zero) and a general maintenance item that almost never
is. Which one an inspector picks decides whether the unit is zeroed.

In [ ]:
PAIRS = {
    "air conditioning": (r"^air conditioning not in operating", r"^provide and maintain, in operating condition, air conditioning"),
    "heating": (r"^heating equipment not in operating", r"^provide and maintain, in operating condition, heating"),
    "hot water": (r"^no hot water", r"^maintain all water heating equipment: in operating condition"),
}
fy_of = insp.set_index("sr_number").fiscal_year
rows, by_insp = [], []
for cond, (specific, general) in PAIRS.items():
    sp_ = cited.subject.str.contains(specific, case=False, regex=True)
    ge_ = cited.subject.str.contains(general, case=False, regex=True)
    c = cited[sp_ | ge_].assign(specific=sp_[sp_ | ge_]).merge(insp[["sr_number", "inspector_id"]], on="sr_number")
    c["fy"] = c.sr_number.map(fy_of)
    rows.append({"condition": cond, "citations": len(c), "share using the life-hazard item": c.specific.mean(),
                 **{fy: c.loc[c.fy == fy, "specific"].mean() for fy in sorted(c.fy.dropna().unique())}})
    by_insp.append(c.assign(condition=cond))
print(pd.DataFrame(rows).round(2).to_string(index=False))

allc = pd.concat(by_insp)
g = allc.groupby("inspector_id").specific.agg(["size", "mean"])
g = g[g["size"] >= 5]
print(f"\ninspectors with 5+ AC/heat/hot-water citations: {len(g)}; share using the life-hazard item, "
      f"percentiles 10/50/90: {g['mean'].quantile([.1, .5, .9]).round(2).tolist()}; "
      f"always general item: {(g['mean'] == 0).sum()}; always life-hazard item: {(g['mean'] == 1).sum()}")
both_used = allc.groupby("fy").inspector_id.nunique()
print("inspectors citing these conditions per year:", both_used.to_dict())

## 4. Hazard conditions noted but not flagged

In [ ]:
PATTERNS = {
    "no heat": r"\bno heat|heat(er|ing)?\s+(is\s+)?(not\s+work|inop)|without heat",
    "no AC": r"\bno\s+(a/?c|air)\b|\b(a/?c|air condition\w*)\s+(is\s+)?(not\s+(work|cool|blow)|inop)|without (a/?c|air)",
    "no hot water": r"no hot water|without hot water",
    "sewage": r"sewage|sewer\s+(leak|back|overflow)|raw sew",
    "exposed wiring": r"exposed\s+(electrical\s+)?wir|live\s+wire",
}
note = cited.comments.fillna("")
rows = []
for k, p in PATTERNS.items():
    m = note.str.contains(p, case=False, regex=True)
    rows.append({"condition": k, "citations with this note": int(m.sum()),
                 "inspections": cited.loc[m, "sr_number"].nunique(),
                 "flagged life hazard": cited.loc[m, "life_hazard"].mean()})
print(pd.DataFrame(rows).to_string(index=False))

any_hz = note.str.contains("|".join(f"(?:{p})" for p in PATTERNS.values()), case=False, regex=True)
per_sr = cited.assign(hz=any_hz).groupby("sr_number").agg(hz=("hz", "max"), lh=("life_hazard", "max"))
hz_sr = per_sr[per_sr.hz]
print(f"\ninspections with a hazard-condition note: {len(hz_sr)}; with no life hazard flagged anywhere "
      f"in the inspection: {(hz_sr.lh == 0).mean():.1%}")
sc = insp.set_index("sr_number").score
print(f"  their median score: {sc.reindex(hz_sr.index).median():.1f}; share scoring 90+: "
      f"{(sc.reindex(hz_sr[hz_sr.lh == 0].index) >= 90).mean():.1%} of those with no flag")